# dx-local v2 — E3 Bio_ClinicalBERT (GPU)
Attach ddxplus-v2 dataset (processed_*.csv + v2_*.csv + release_*_v2.json). Accelerator: GPU T4/P100, internet ON.
Fine-tunes Bio_ClinicalBERT on rendered narratives over the merged 309-class corpus. Outputs `artifacts_e3.zip`.


In [ ]:
!pip -q install 'transformers[torch]' datasets accelerate


In [ ]:
# GPU compatibility probe BEFORE torch is imported anywhere
import subprocess, sys
chk = "import torch; t=torch.ones(8,device='cuda'); (t*2).sum().item()"
probe = subprocess.run([sys.executable, '-c', chk], capture_output=True)
if probe.returncode == 0:
    print('stock torch OK on this GPU')
else:
    print('GPU incompatible with stock torch; installing cu121 build...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.6.0',
         'torchvision==0.21.0', 'torchaudio==2.6.0',
         '--index-url', 'https://download.pytorch.org/whl/cu124'],
        check=True,
    )
    probe2 = subprocess.run([sys.executable, '-c', chk], capture_output=True)
    print('torch 2.6.0+cu124 reinstalled; probe2 rc =', probe2.returncode)
    assert probe2.returncode == 0, 'even cu124 torch cannot run on this GPU'


In [ ]:
import ast, hashlib, json, math, os, time, zipfile, zlib
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

SEED = 20260822
BASE = 'emilyalsentzer/Bio_ClinicalBERT'
IN = '/kaggle/input'
FILES_V1 = ('processed_train.csv', 'processed_validate.csv', 'processed_test.csv')
FILES_V2 = ('v2_train.csv', 'v2_validate.csv', 'v2_test.csv')

def _locate():
    wanted = {'processed_train.csv'} | set(FILES_V2)
    for dp, _, fs in os.walk(IN):
        if wanted <= set(fs):
            return dp
    return None

DDX = None
for attempt in range(60):
    DDX = _locate()
    if DDX:
        break
    if attempt == 0:
        print('waiting for dataset mount; saw:', os.listdir(IN))
    time.sleep(10)
if DDX is None:
    raise RuntimeError('dataset never mounted')
print('dataset dir:', DDX)


In [ ]:
cols = ['PATHOLOGY', 'AGE', 'SEX', 'EVIDENCES']
frames = [pd.read_csv(f'{DDX}/{f}', dtype={'PATHOLOGY': 'string'})[cols]
          for f in (*FILES_V1, *FILES_V2)]
pool = pd.concat(frames, ignore_index=True); del frames
pool['AGE'] = pool['AGE'].fillna(-1).astype(int)
key = (pool['PATHOLOGY'].astype(str) + '|' + pool['AGE'].astype(str) + '|'
       + pool['SEX'].astype(str) + '|' + pool['EVIDENCES'].astype(str))
pool = pool.loc[~key.duplicated()].reset_index(drop=True)

parts = {'train': [], 'validate': []}
for pathology, group in pool.groupby('PATHOLOGY', sort=False):
    gseed = (SEED + zlib.crc32(str(pathology).encode())) % 2**32
    idx = np.random.default_rng(gseed).permutation(len(group))
    n = len(group); c1 = int(n*0.8); c2 = int(n*0.9)
    parts['train'].append(group.iloc[np.sort(idx[:c1])])
    parts['validate'].append(group.iloc[np.sort(idx[c1:c2])])
del pool
splits = {s: pd.concat(v, ignore_index=True) for s, v in parts.items()}
print({k: len(v) for k, v in splits.items()})


In [ ]:
PREFIXES = ('do you have ', 'have you ', 'do you feel ', 'are you ', 'did you ',
            'does the person have ', 'does the ', 'is there ')
onto_e = json.load(open(f'{DDX}/release_evidences_v2.json'))

def render(row):
    age = row['AGE']; sex = row['SEX']
    who = 'A patient.'
    if age == age and age > 0 and sex in ('M', 'F'):
        who = f'A {int(age)}-year-old {"man" if sex=="M" else "woman"}.'
    parts_ = [who]
    for c in sorted(set(ast.literal_eval(row['EVIDENCES'] or '[]'))):
        b = c.split('_@_')[0]
        ent = onto_e.get(b)
        if ent is None:
            continue
        q = ent['question_en'].rstrip('?').strip()
        ql = q.lower()
        for pre in PREFIXES:
            if ql.startswith(pre):
                q = q[len(pre):].strip(); break
        if q:
            parts_.append(q + '.')
    return ' '.join(parts_)

train_df = splits['train']; val_df = splits['validate']
labels = sorted(train_df['PATHOLOGY'].unique())
lut = {c: i for i, c in enumerate(labels)}
json.dump({'pathologies': labels}, open('labels.json', 'w'))

CAP_PER_CLASS = 600
train_df = (train_df.groupby('PATHOLOGY', group_keys=False)
            .apply(lambda g: g.sample(min(len(g), CAP_PER_CLASS), random_state=SEED)))
val_cap = min(len(val_df), 25_000)
val_df = val_df.sample(val_cap, random_state=SEED)
train_texts = [render(r) for _, r in train_df.iterrows()]
train_y = [lut[p] for p in train_df['PATHOLOGY']]
val_texts = [render(r) for _, r in val_df.iterrows()]
val_y = [lut[p] for p in val_df['PATHOLOGY']]
print(len(train_texts), 'train /', len(val_texts), 'val;', len(labels), 'classes')
print('sample:', train_texts[0][:200])


In [ ]:
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=len(labels))

enc_t = tok(train_texts, truncation=True, max_length=128, padding='max_length', return_tensors='pt')
enc_v = tok(val_texts, truncation=True, max_length=128, padding='max_length', return_tensors='pt')

class DS(torch.utils.data.Dataset):
    def __init__(self, enc, ys):
        self.enc, self.ys = enc, ys
    def __len__(self):
        return len(self.ys)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item['labels'] = torch.tensor(self.ys[i])
        return item

args = TrainingArguments(
    output_dir='/kaggle/working/e3_ckpt',
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    learning_rate=5e-5,
    num_train_epochs=2,
    weight_decay=0.01,
    warmup_ratio=0.06,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    seed=SEED,
    report_to=[],
)

def compute_metrics(eval_pred):
    logits, yy = eval_pred
    preds = np.asarray(logits).argmax(1)
    return {'accuracy': float((preds == np.asarray(yy)).mean())}

trainer = Trainer(model=model, args=args, train_dataset=DS(enc_t, train_y),
                  eval_dataset=DS(enc_v, val_y), compute_metrics=compute_metrics)
metrics = trainer.evaluate()
trainer.train()
metrics = trainer.evaluate()
print(metrics)


In [ ]:
out = '/kaggle/working/artifacts_e3/'
os.makedirs(out, exist_ok=True)
trainer.save_model(out + 'e3_bert/')
tok.save_pretrained(out + 'e3_bert/')
json.dump(json.load(open('labels.json')), open(out + 'labels.json', 'w'))
json.dump({k: float(v) for k, v in metrics.items()}, open(out + 'metrics.json', 'w'))
manifest = {}
for root_, _, files in os.walk(out):
    for f in files:
        full = os.path.join(root_, f)
        manifest[full.replace(out, '')] = hashlib.sha256(open(full, 'rb').read()).hexdigest()
json.dump(manifest, open(out + 'MANIFEST.sha256.json', 'w'), indent=1)
with zipfile.ZipFile('/kaggle/working/artifacts_e3.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for root_, _, files in os.walk(out):
        for f in files:
            full = os.path.join(root_, f)
            z.write(full, full.replace(out, ''))
print('artifacts_e3.zip written:',
      os.path.getsize('/kaggle/working/artifacts_e3.zip') // 1024 // 1024, 'MB')
